# SFT Actor Training - Behavior Cloning (GPU Accelerated)

Trains the APActor network via supervised learning on expert demonstrations.
- **5000 epochs** for thorough training
- **GPU accelerated** using CUDA
- Includes training visualization and evaluation


In [14]:
import json
import torch
import torch.nn as nn
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from copy import deepcopy
from torch.distributions.categorical import Categorical
from pathlib import Path

# Check GPU availability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Memory: 8.18 GB


In [15]:
# Import actor model
import sys
sys.path.insert(0, '.')

from actor_wifi import APActor, APGraphBatch


## Configuration


In [24]:
# Training Configuration
CONFIG = {
    # Dataset path - UPDATE THIS to your dataset
    'dataset_path': 'merged_final.json',
    
    # Training parameters - 5000 epochs for thorough training
    'epochs': 5000,
    'batch_size': 64,
    'lr': 1e-3,
    'entropy_coef': 0.01,  # Small entropy bonus for exploration
    'clip_value': 4.0,
    'warmup_steps': 0,
    'seed': 42,
    
    # Model architecture
    'input_dim_ap': 4,
    'hidden_dim': 128,
    'gnn_layers': 2,
    'mlp_layers': 3,
    'num_channels': 3,
    'num_power_levels': 7,
    'heads': 1,
    'dropout': 0.0,
    
    # Output
    'save_path': './actor_sft_sisa.pth',
    'log_interval': 100,  # Print every N epochs
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


Configuration:
  dataset_path: merged_final.json
  epochs: 5000
  batch_size: 64
  lr: 0.001
  entropy_coef: 0.01
  clip_value: 4.0
  warmup_steps: 0
  seed: 42
  input_dim_ap: 4
  hidden_dim: 128
  gnn_layers: 2
  mlp_layers: 3
  num_channels: 3
  num_power_levels: 7
  heads: 1
  dropout: 0.0
  save_path: ./actor_sft_sisa.pth
  log_interval: 100


## Helper Functions


In [25]:
def set_seed(seed):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Channel width to action index mapping
CHANNEL_WIDTH_TO_ACTION = {20: 0, 40: 1, 80: 2}

# Power levels mapping (7 levels from 12 to 24 dBm in 2 dBm steps)
POWER_LEVELS = [12, 14, 16, 18, 20, 22, 24]

def power_to_action(power):
    """Convert power value to nearest action index (0-6)."""
    power = float(power)
    # Find closest power level
    min_dist = float('inf')
    best_idx = 3  # default to middle
    for i, p in enumerate(POWER_LEVELS):
        dist = abs(power - p)
        if dist < min_dist:
            min_dist = dist
            best_idx = i
    return best_idx

def load_dataset(path):
    """Load and transform dataset from merged_final.json format."""
    with open(path, 'r') as f:
        raw_data = json.load(f)
    
    # Get the data array from the JSON structure
    if isinstance(raw_data, dict) and 'data' in raw_data:
        samples = raw_data['data']
    else:
        samples = raw_data
    
    transformed = []
    for sample in samples:
        try:
            X = sample['X']
            Y = sample['Y']
            
            # Build AP features: [channel, power, client_count, channel_util]
            ap_features = []
            ap_ids = sorted(X['aps'].keys())  # AP1, AP2, AP3, AP4
            for ap_id in ap_ids:
                ap = X['aps'][ap_id]
                features = [
                    float(ap['CHANNEL']) / 200.0,           # Normalize channel
                    float(ap['AP_POWER']) / 24.0,           # Normalize power
                    float(ap.get('Client_count', 5)) / 10.0, # Normalize clients
                    float(ap.get('Channel_util', 0.5))       # Already 0-1
                ]
                ap_features.append(features)
            
            # Build edge_index from RSSI matrix (connect APs with RSSI > -100)
            rssi_matrix = X.get('rssi_matrix', [])
            num_aps = len(ap_ids)
            src, dst = [], []
            if rssi_matrix:
                for i in range(num_aps):
                    for j in range(num_aps):
                        if i != j and rssi_matrix[i][j] > -100:
                            src.append(i)
                            dst.append(j)
            
            # If no edges from RSSI, create fully connected graph
            if not src:
                for i in range(num_aps):
                    for j in range(num_aps):
                        if i != j:
                            src.append(i)
                            dst.append(j)
            
            edge_index = [src, dst]
            
            # Get target AP index
            target_ap_idx = sample.get('changed_ap_index', Y.get('ap_index', 0))
            
            # Get channel width action (0=20MHz, 1=40MHz, 2=80MHz)
            channel_width = Y.get('CHANNEL_WIDTH_MHZ', 20)
            channel_action = CHANNEL_WIDTH_TO_ACTION.get(channel_width, 0)
            
            # Get power action (0-6)
            power_action = power_to_action(Y.get('AP_POWER', 18))
            
            transformed.append({
                'state': {
                    'ap_features': ap_features,
                    'edge_index': edge_index,
                    'target_ap_idx': target_ap_idx
                },
                'channel_action': channel_action,
                'power_action': power_action
            })
        except Exception as e:
            print(f"Skipping sample due to error: {e}")
            continue
    
    print(f"Loaded {len(transformed)} training samples from {path}")
    return transformed

def sample_to_graph_batch(sample):
    """Convert a single dataset sample to APGraphBatch."""
    state = sample['state']
    ap_features = np.array(state['ap_features'], dtype=np.float32)
    edge_index = np.array(state['edge_index'], dtype=np.int64)
    target_ap_idx = state['target_ap_idx']
    batch = APGraphBatch()
    batch.from_single(ap_features, edge_index, target_ap_idx)
    return batch

def get_action_logits(actor, batch_states):
    """Get raw logits from actor for BC training."""
    edge_index = torch.cat([batch_states.edge_index, batch_states.edge_index.flip(0)], dim=-1)
    ap_embeddings = actor.gat(batch_states.ap_features, edge_index)
    target_embed = ap_embeddings[batch_states.target_ap_idx]
    channel_logits = actor.channel_head(target_embed)
    power_logits = actor.power_head(target_embed)
    channel_dist = Categorical(logits=channel_logits)
    power_dist = Categorical(logits=power_logits)
    entropy = channel_dist.entropy() + power_dist.entropy()
    return channel_logits, power_logits, entropy


## Load Dataset


In [26]:
# Set seed and load dataset
set_seed(CONFIG['seed'])
dataset = load_dataset(CONFIG['dataset_path'])

# Initialize model on GPU
actor = APActor(
    input_dim_ap=CONFIG['input_dim_ap'],
    hidden_dim=CONFIG['hidden_dim'],
    gnn_layers=CONFIG['gnn_layers'],
    mlp_layers=CONFIG['mlp_layers'],
    num_channels=CONFIG['num_channels'],
    num_power_levels=CONFIG['num_power_levels'],
    heads=CONFIG['heads'],
    dropout=CONFIG['dropout']
).to(device)

print(f"Model on {device}, params: {sum(p.numel() for p in actor.parameters()):,}")


Loaded 577 training samples from merged_final.json
Model on cuda, params: 85,002


## Training Loop (5000 epochs)


In [28]:
# Training setup
optimizer = torch.optim.Adam(actor.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)
criterion_channel = nn.CrossEntropyLoss()
criterion_power = nn.CrossEntropyLoss()

valid_indices = list(range(len(dataset)))
history = {'channel_loss': [], 'power_loss': [], 'total_loss': [], 'entropy': []}
best_loss = float('inf')

print(f"Training for {CONFIG['epochs']} epochs on {len(valid_indices)} samples...")
total_start = time.time()

for epoch in range(CONFIG['epochs']):
    actor.train()
    indices = np.random.permutation(valid_indices)
    epoch_ch, epoch_pwr, epoch_ent = [], [], []
    
    for start_idx in range(0, len(indices), CONFIG['batch_size']):
        batch_indices = indices[start_idx:start_idx + CONFIG['batch_size']]
        temp_states = [deepcopy(sample_to_graph_batch(dataset[k])) for k in batch_indices]
        
        channel_actions = torch.tensor([dataset[k]['channel_action'] for k in batch_indices], dtype=torch.long, device=device)
        power_actions = torch.tensor([dataset[k]['power_action'] for k in batch_indices], dtype=torch.long, device=device)
        
        batch_states = APGraphBatch().batch_process(temp_states)
        channel_logits, power_logits, entropy = get_action_logits(actor, batch_states)
        
        ch_loss = criterion_channel(channel_logits, channel_actions)
        pwr_loss = criterion_power(power_logits, power_actions)
        loss = ch_loss + pwr_loss - CONFIG['entropy_coef'] * entropy.mean()
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
        optimizer.step()
        
        epoch_ch.append(ch_loss.item())
        epoch_pwr.append(pwr_loss.item())
        epoch_ent.append(entropy.mean().item())
    
    scheduler.step()
    mean_total = np.mean(epoch_ch) + np.mean(epoch_pwr)
    history['channel_loss'].append(np.mean(epoch_ch))
    history['power_loss'].append(np.mean(epoch_pwr))
    history['total_loss'].append(mean_total)
    history['entropy'].append(np.mean(epoch_ent))
    
    if mean_total < best_loss:
        best_loss = mean_total
        torch.save(actor.state_dict(), CONFIG['save_path'])
    
    if (epoch + 1) % CONFIG['log_interval'] == 0:
        elapsed = (time.time() - total_start) / 60
        print(f"Epoch {epoch+1:5d}: loss={mean_total:.4f}, best={best_loss:.4f}, time={elapsed:.1f}min")

print(f"\nTraining complete! Best loss: {best_loss:.4f}, saved to {CONFIG['save_path']}")


Training for 5000 epochs on 577 samples...
Epoch   100: loss=0.1282, best=0.0590, time=0.3min
Epoch   200: loss=0.1702, best=0.0590, time=0.5min
Epoch   300: loss=0.1766, best=0.0418, time=0.8min
Epoch   400: loss=0.0519, best=0.0418, time=1.0min


KeyboardInterrupt: 

## Training Visualization


In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total Loss with smoothing
ax = axes[0, 0]
ax.plot(history['total_loss'], 'b-', alpha=0.3)
window = min(100, len(history['total_loss']) // 10)
if window > 1:
    smoothed = np.convolve(history['total_loss'], np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(history['total_loss'])), smoothed, 'b-', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss'); ax.set_title('Total Loss'); ax.grid(True, alpha=0.3)

# Channel vs Power Loss
ax = axes[0, 1]
ax.plot(history['channel_loss'], 'r-', alpha=0.7, label='Channel')
ax.plot(history['power_loss'], 'g-', alpha=0.7, label='Power')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Channel vs Power Loss'); ax.legend(); ax.grid(True, alpha=0.3)

# Entropy
ax = axes[1, 0]
ax.plot(history['entropy'], 'purple', alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('Entropy'); ax.set_title('Policy Entropy'); ax.grid(True, alpha=0.3)

# Log scale loss
ax = axes[1, 1]
ax.semilogy(history['total_loss'], 'b-', alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log)'); ax.set_title('Total Loss (Log Scale)'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_5000.png', dpi=150)
plt.show()


## Model Evaluation


In [ ]:
# Load best model and evaluate accuracy
actor.load_state_dict(torch.load(CONFIG['save_path'], map_location=device))
actor.eval()

correct_ch, correct_pwr, total = 0, 0, 0
with torch.no_grad():
    for i in range(min(1000, len(dataset))):
        batch = sample_to_graph_batch(dataset[i])
        batch_states = APGraphBatch().batch_process([batch])
        ch_logits, pwr_logits, _ = get_action_logits(actor, batch_states)
        
        if ch_logits.argmax(dim=1).item() == dataset[i]['channel_action']:
            correct_ch += 1
        if pwr_logits.argmax(dim=1).item() == dataset[i]['power_action']:
            correct_pwr += 1
        total += 1

print(f"Accuracy on {total} samples:")
print(f"  Channel: {100*correct_ch/total:.2f}%")
print(f"  Power:   {100*correct_pwr/total:.2f}%")

# Save history
with open('training_history_5000.json', 'w') as f:
    json.dump(history, f)
print(f"\nHistory saved to training_history_5000.json")
